# GPT-2 の動作原理を数式レベルで確認する

<a href="https://colab.research.google.com/github/miyoshi-nii/llm_test/blob/main/docs/gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このノートブックでは、GPT-2（Radford et al., 2019, *Language Models are Unsupervised Multitask Learners*）を **NumPy だけで実装** し、OpenAI が公開した **学習済みの重み（GPT-2 small, 124M パラメータ）** を読み込んで、実際のモデルの内部で何が起きているかを数値で確認します。最後に、GPT-2 の系譜から生まれた **スケーリング則**（Kaplan et al., 2020 / Hoffmann et al., 2022）を扱います。

- **学習・評価は行いません**（推論のみ）。
- Transformer の基本部品（Attention・LayerNorm・FFN・位置エンコーディング・KV キャッシュ）は [transformer.ipynb](transformer.ipynb) で確認済みという前提で、ここでは **GPT-2 に固有の点** と **学習済みモデルでしか見られない現象** に重点を置きます。
- 自前実装の出力が Hugging Face `transformers` の実装と一致することも確かめます。

### GPT-2 small の諸元

| 記号 | 意味 | 値 |
|---|---|---|
| $d$ | 隠れ状態の次元 `n_embd` | 768 |
| $H$ | ヘッド数 `n_head` | 12 |
| $d_h = d/H$ | 1 ヘッドの次元 | 64 |
| $L$ | ブロック（層）数 `n_layer` | 12 |
| $n_{\text{ctx}}$ | 最大系列長 `n_positions` | 1024 |
| $\lvert\mathcal{V}\rvert$ | 語彙サイズ | 50257 |

### 原論文の Transformer（2017）との違い

| | Transformer (Vaswani et al.) | GPT-2 |
|---|---|---|
| 構成 | Encoder–Decoder | **Decoder のみ**（Cross-Attention なし） |
| 正規化 | Post-LN：$\mathrm{LN}(x + f(x))$ | **Pre-LN**：$x + f(\mathrm{LN}(x))$、最後に `ln_f` |
| 位置情報 | 正弦波（固定） | **学習可能な位置埋め込み** $W_{pe}\in\mathbb{R}^{1024\times 768}$ |
| 活性化関数 | ReLU | **GELU**（tanh 近似） |
| トークナイザ | BPE | **バイトレベル BPE**（未知語なし） |
| 出力層 | 埋め込みと共有 | 埋め込み $W_{te}$ と共有 |
| 埋め込みのスケール | $\times\sqrt{d}$ | なし |
| 初期化 | — | 残差に足し込む射影の重みを $1/\sqrt{2L}$ 倍（学習時のみ関係） |

ベクトルは前回と同じく **行ベクトル**、系列は $X\in\mathbb{R}^{T\times d}$（1 行 = 1 トークン）とします。

### 目次
0. 準備
1. 学習済み重みの読み込みとパラメータの内訳
2. バイトレベル BPE トークナイザ
3. 埋め込み（トークン埋め込み + 学習済み位置埋め込み）
4. Transformer ブロック（Pre-LN）を 1 ステップずつ
5. モデル全体の順伝播と `transformers` との一致確認
6. 次トークン予測と言語モデルの確率
7. 注意パターンの観察（attention sink・直前トークンヘッド・induction head）
8. Logit lens：層を追って予測が形成される様子
9. 残差ストリームのノルム
10. 生成：KV キャッシュとデコード戦略
11. スケーリング則
12. まとめと参考文献

## 0. 準備

### Google Colab で実行する場合
- 上の **Open in Colab** バッジから開く（GitHub の `main` ブランチに push されている必要があります）か、「ファイル → ノートブックをアップロード」で `gpt2.ipynb` を選びます。
- 必要なライブラリ（`numpy`, `matplotlib`, `regex`, `huggingface_hub`, `safetensors`, `transformers`, `torch`）はすべて Colab に入っているので、追加インストールは不要です。GPU も不要です（CPU ランタイムで動きます）。
- 初回実行時に Hugging Face Hub から GPT-2 の重み（約 550 MB）をダウンロードします。メモリは 2 GB 程度あれば足ります。

### ローカル（このリポジトリの `.venv`）で実行する場合
```bash
uv pip install numpy matplotlib regex huggingface_hub safetensors ipykernel
# transformers との一致確認（第 2・5 節）も行う場合
uv pip install torch transformers
```
`torch` / `transformers` が無くても、一致確認のセルがスキップされるだけで他はすべて動きます。

In [ ]:
import json
import math
import sys
import time

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import regex                                # \p{L} などの Unicode プロパティを使うため（標準の re では不可）
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file     # torch なしで safetensors を NumPy 配列として読む

IN_COLAB = "google.colab" in sys.modules
np.set_printoptions(precision=4, suppress=True, linewidth=120)
rng = np.random.default_rng(0)
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカル'} / Python {sys.version.split()[0]}"
      f" / numpy {np.__version__} / matplotlib {matplotlib.__version__}")


def heatmap(M, title="", xlabels=None, ylabels=None, ax=None, annotate=True, cmap="viridis", fmt="{:.2f}"):
    """小さな行列を数値付きのヒートマップで描く"""
    M = np.asarray(M)
    if ax is None:
        _, ax = plt.subplots(figsize=(0.7 * M.shape[1] + 2.5, 0.55 * M.shape[0] + 1.5))
    im = ax.imshow(M, cmap=cmap, aspect="auto")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)), xlabels, rotation=60, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)), ylabels)
    if annotate and M.size <= 150:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=7,
                        color="w" if im.norm(M[i, j]) < 0.5 else "k")
    plt.colorbar(im, ax=ax, fraction=0.046)
    return ax

## 1. 学習済み重みの読み込みとパラメータの内訳

Hugging Face Hub の [`openai-community/gpt2`](https://huggingface.co/openai-community/gpt2) から、重み（`model.safetensors`）・設定（`config.json`）・トークナイザの辞書（`vocab.json`, `merges.txt`）を取得します。

GPT-2 の重み行列は OpenAI の元実装（`Conv1D`）に由来して **(入力次元, 出力次元)** の形で保存されています。したがって行ベクトル表記の $XW$ にそのまま使えます。

In [ ]:
REPO = "openai-community/gpt2"      # GPT-2 small (124M)
config = json.load(open(hf_hub_download(REPO, "config.json")))
weights = load_file(hf_hub_download(REPO, "model.safetensors"))    # dict[str, np.ndarray]

d_model = config["n_embd"]          # 768
n_heads = config["n_head"]          # 12
n_layers = config["n_layer"]        # 12
n_ctx = config["n_positions"]       # 1024
vocab_size = config["vocab_size"]   # 50257
ln_eps = config["layer_norm_epsilon"]
d_head = d_model // n_heads         # 64
print(f"d_model={d_model}, n_heads={n_heads}, d_head={d_head}, n_layers={n_layers}, "
      f"n_ctx={n_ctx}, vocab_size={vocab_size}, activation={config['activation_function']}")
print(f"\nテンソル数: {len(weights)}。以下はそのうち埋め込み・ブロック 0・最終 LN のもの")
for name in ["wte.weight", "wpe.weight",
             "h.0.ln_1.weight", "h.0.ln_1.bias", "h.0.attn.c_attn.weight", "h.0.attn.c_attn.bias",
             "h.0.attn.c_proj.weight", "h.0.attn.c_proj.bias", "h.0.ln_2.weight", "h.0.ln_2.bias",
             "h.0.mlp.c_fc.weight", "h.0.mlp.c_fc.bias", "h.0.mlp.c_proj.weight", "h.0.mlp.c_proj.bias",
             "h.0.attn.bias", "ln_f.weight", "ln_f.bias"]:
    print(f"  {name:24s} {str(weights[name].shape):22s} {weights[name].dtype}")

`h.{l}.attn.bias` は名前に反して学習パラメータではなく、**因果マスク（下三角行列）を保存したバッファ** です。中身を確認してから除外します。

In [ ]:
buf = weights["h.0.attn.bias"][0, 0]
assert np.array_equal(buf, np.tril(np.ones((n_ctx, n_ctx), dtype=buf.dtype)))
print("✔ h.*.attn.bias は 1024×1024 の下三角行列（因果マスク）→ パラメータから除外")
params = {k: v for k, v in weights.items() if k.split(".")[-2:] != ["attn", "bias"]}
print(f"学習パラメータのテンソル数: {len(params)}")

### パラメータ数の内訳

1 ブロックあたりのパラメータ数は（$d = 768$）

| 部品 | 形状 | パラメータ数 |
|---|---|---|
| `ln_1`（$\gamma,\beta$） | $d,\ d$ | $2d$ |
| `attn.c_attn`（$W_{QKV}, b_{QKV}$） | $d\times 3d,\ 3d$ | $3d^2 + 3d$ |
| `attn.c_proj`（$W_O, b_O$） | $d\times d,\ d$ | $d^2 + d$ |
| `ln_2`（$\gamma,\beta$） | $d,\ d$ | $2d$ |
| `mlp.c_fc`（$W_{fc}, b_{fc}$） | $d\times 4d,\ 4d$ | $4d^2 + 4d$ |
| `mlp.c_proj`（$W_{proj}, b_{proj}$） | $4d\times d,\ d$ | $4d^2 + d$ |
| **合計** | | $12d^2 + 13d$ |

これに埋め込み $W_{te}$（$\lvert\mathcal{V}\rvert d$）、位置埋め込み $W_{pe}$（$n_{\text{ctx}} d$）、最終 LN（$2d$）が加わります。

In [ ]:
d = d_model
rows = [
    ("ln_1 (γ, β)",         ["ln_1.weight", "ln_1.bias"],                 2 * d),
    ("attn.c_attn (Q,K,V)", ["attn.c_attn.weight", "attn.c_attn.bias"],   3 * d * d + 3 * d),
    ("attn.c_proj (O)",     ["attn.c_proj.weight", "attn.c_proj.bias"],   d * d + d),
    ("ln_2 (γ, β)",         ["ln_2.weight", "ln_2.bias"],                 2 * d),
    ("mlp.c_fc",            ["mlp.c_fc.weight", "mlp.c_fc.bias"],         4 * d * d + 4 * d),
    ("mlp.c_proj",          ["mlp.c_proj.weight", "mlp.c_proj.bias"],     4 * d * d + d),
]
print("1 ブロックあたり:")
n_block = 0
for name, keys, formula in rows:
    n = sum(params[f"h.0.{k}"].size for k in keys)
    assert n == formula
    n_block += n
    print(f"  {name:22s} {n:>12,}")
assert n_block == 12 * d * d + 13 * d
print(f"  {'合計 (12d² + 13d)':22s} {n_block:>12,}\n")

n_wte = params["wte.weight"].size
n_wpe = params["wpe.weight"].size
n_lnf = params["ln_f.weight"].size + params["ln_f.bias"].size
n_total = sum(v.size for v in params.values())
print(f"  wte  (トークン埋め込み・出力層と共有) {n_wte:>12,}  = |V|·d")
print(f"  wpe  (位置埋め込み)                 {n_wpe:>12,}  = n_ctx·d")
print(f"  ブロック × {n_layers}                      {n_layers * n_block:>12,}")
print(f"  ln_f                                {n_lnf:>12,}")
print(f"  合計                                {n_total:>12,}")
assert n_total == n_wte + n_wpe + n_layers * n_block + n_lnf

N_nonemb = n_total - n_wte - n_wpe
print(f"\n埋め込みを除くパラメータ数 N = {N_nonemb:,}  （近似式 12·L·d² = {12 * n_layers * d * d:,}）"
      "  ← 第 11 節のスケーリング則で使う")

In [ ]:
def get_block(l):
    g = lambda name: params[f"h.{l}.{name}"]
    return {"ln1_g": g("ln_1.weight"), "ln1_b": g("ln_1.bias"),
            "W_qkv": g("attn.c_attn.weight"), "b_qkv": g("attn.c_attn.bias"),
            "W_o": g("attn.c_proj.weight"), "b_o": g("attn.c_proj.bias"),
            "ln2_g": g("ln_2.weight"), "ln2_b": g("ln_2.bias"),
            "W_fc": g("mlp.c_fc.weight"), "b_fc": g("mlp.c_fc.bias"),
            "W_proj": g("mlp.c_proj.weight"), "b_proj": g("mlp.c_proj.bias")}


blocks = [get_block(l) for l in range(n_layers)]
wte, wpe = params["wte.weight"], params["wpe.weight"]
lnf_g, lnf_b = params["ln_f.weight"], params["ln_f.bias"]

## 2. バイトレベル BPE トークナイザ

GPT-2 のトークナイザは **バイトレベル BPE**（Byte Pair Encoding）です。

- 基本単位は **UTF-8 の 256 種類のバイト**。どんな文字列でもバイト列にはできるので、**未知語（`<unk>`）が存在しません**。
- 学習データ中で頻出する隣接ペアを順に結合した **50,000 個の結合規則**（`merges.txt`）を持ちます。
- 語彙数は $256 + 50000 + 1$（`<|endoftext|>`）$= 50257$。

エンコードの手順：
1. 正規表現で文字列を「単語」に **事前分割**（pre-tokenize）する。
2. 各単語を UTF-8 バイト列にし、各バイトを **印字可能な Unicode 文字に写像** する（`bytes_to_unicode`）。
3. 記号列 $(s_1, \dots, s_n)$ の隣接ペアのうち、**結合規則の順位（rank）が最も小さい（＝学習の早い段階で採用された）ペア** $(a, b) = \arg\min_{(s_i, s_{i+1})} \mathrm{rank}(s_i, s_{i+1})$ を選び、すべての出現箇所で $ab$ に結合する。結合できるペアがなくなるまで繰り返す。
4. 得られた各記号を `vocab.json` で ID に変換する。

In [ ]:
encoder = json.load(open(hf_hub_download(REPO, "vocab.json"), encoding="utf-8"))   # トークン文字列 → ID
decoder = {i: s for s, i in encoder.items()}                                       # ID → トークン文字列
merge_lines = open(hf_hub_download(REPO, "merges.txt"), encoding="utf-8").read().split("\n")
merges = [tuple(m.split()) for m in merge_lines[1:] if m]          # 先頭行 "#version: 0.2" を除く
bpe_ranks = {pair: r for r, pair in enumerate(merges)}             # ペア → 順位（小さいほど優先）
EOT = encoder["<|endoftext|>"]

print(f"語彙数 {len(encoder)} = 256（バイト） + {len(merges)}（結合規則） + 1（<|endoftext|> = {EOT}）")
assert len(encoder) == 256 + len(merges) + 1 == vocab_size
print("最初に学習された結合規則 10 個:", merges[:10])

### バイト → 印字可能文字への写像

空白や制御文字（0x00〜0x20 など）をそのまま語彙の文字列にすると扱いにくいので、GPT-2 は 256 個のバイトを **すべて印字可能な Unicode 文字に 1 対 1 で写像** します。印字可能な ASCII などはそのまま、それ以外は $256 + n$ 番目の文字に移します。その結果、**空白（0x20）は `Ġ`（U+0120）、改行（0x0A）は `Ċ`（U+010A）** になります。語彙中のトークン `Ġthe` は「空白 + the」という意味です。

In [ ]:
def bytes_to_unicode():
    """256 個のバイト → 印字可能な Unicode 文字（GPT-2 の encoder.py と同じ規則）"""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:          # 印字できないバイトは 256 + n 番目の文字に移す
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))


byte_encoder = bytes_to_unicode()
byte_decoder = {c: b for b, c in byte_encoder.items()}
assert len(set(byte_encoder.values())) == 256          # 1 対 1
for b in [ord("A"), ord(" "), ord("\n"), 0xE3]:
    print(f"byte 0x{b:02X} ({chr(b)!r:6s}) → {byte_encoder[b]!r} (U+{ord(byte_encoder[b]):04X})")

### 事前分割（pre-tokenize）

GPT-2 は次の正規表現で文字列を分割してから、単語ごとに BPE を適用します。英語の短縮形（`'s`, `'ll` など）、**直前の空白 1 個を含めた** 文字の連続・数字の連続・記号の連続、空白、に分けます。単語の境界をまたぐ結合（例：`dog.` の `g` と `.`）が起きないようにするためです。

In [ ]:
PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
print(PAT.findall("Hello world! It's GPT-2's tokenizer:  2024年の日本語も OK."))

### BPE の結合過程を 1 ステップずつ見る

In [ ]:
def bpe(word, verbose=False):
    """word（写像済みの文字列）に BPE の結合規則を順位の小さい順に適用する"""
    symbols = list(word)
    step = 0
    if verbose:
        print(f"初期状態        : {symbols}")
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        best = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if best not in bpe_ranks:          # 結合できるペアがもう無い
            break
        merged, i = [], 0
        while i < len(symbols):            # best の出現箇所をすべて結合
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best:
                merged.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        symbols = merged
        step += 1
        if verbose:
            print(f"step {step:2d} rank {bpe_ranks[best]:5d}: {best[0]!r} + {best[1]!r} → {symbols}")
    return symbols


word = "".join(byte_encoder[b] for b in " tokenization".encode("utf-8"))
print(f"' tokenization' → 写像後 {word!r}\n")
pieces = bpe(word, verbose=True)
print("\n最終結果:", pieces, "→ ID", [encoder[p] for p in pieces])

In [ ]:
def encode(text):
    """文字列 → トークン ID 列（<|endoftext|> などの特殊トークンは扱わない）"""
    ids = []
    for w in PAT.findall(text):
        mapped = "".join(byte_encoder[b] for b in w.encode("utf-8"))
        ids.extend(encoder[p] for p in bpe(mapped))
    return ids


def decode(ids):
    """トークン ID 列 → 文字列（不完全な UTF-8 バイト列は U+FFFD '�' になる）"""
    text = "".join(decoder[int(i)] for i in ids)
    return bytearray(byte_decoder[c] for c in text).decode("utf-8", errors="replace")


def tok_label(i):
    """プロット用：ID → ASCII だけの表示文字列（フォントによる文字化けを防ぐ）"""
    if int(i) == EOT:
        return "<eot>"
    s = decode([i]).replace("\n", "\\n")
    return s.encode("ascii", "backslashreplace").decode("ascii")


samples = ["Hello world! GPT-2 is a language model.",
           "Transformers process 1234567 tokens per second.",
           "こんにちは、世界。言語モデルの仕組みを学ぶ。",
           "emoji 🤖 and   multiple spaces\nnew line"]
for s in samples:
    ids = encode(s)
    assert decode(ids) == s                        # 可逆（情報が失われない）
    print(f"{s!r}\n  → {len(s)} 文字, {len(ids)} トークン: {[decoder[i] for i in ids]}\n")
print("✔ すべての例で decode(encode(s)) == s（バイトレベルなので未知語が無い）")

**観察**：日本語は 1 文字が UTF-8 で 3 バイトになり、語彙に結合規則が少ないため、**1 文字あたりのトークン数が英語より多く** なります。また、1 トークンが文字の途中で切れている場合があり、そのトークンを単独でデコードすると不完全なバイト列（`�`）になります。

In [ ]:
s = "言語モデル"
ids = encode(s)
print(f"{s!r}: UTF-8 で {len(s.encode('utf-8'))} バイト → {len(ids)} トークン")
for i in ids:
    raw = bytes(byte_decoder[c] for c in decoder[i])
    print(f"  ID {i:5d}: bytes={raw.hex(' '):12s} 単独でデコード → {decode([i])!r}")

en = "The quick brown fox jumps over the lazy dog."
ja = "素早い茶色の狐がのろまな犬を飛び越える。"
print(f"\n英語: {len(en)} 文字 / {len(encode(en))} トークン = {len(en) / len(encode(en)):.2f} 文字/トークン")
print(f"日本語: {len(ja)} 文字 / {len(encode(ja))} トークン = {len(ja) / len(encode(ja)):.2f} 文字/トークン")

In [ ]:
# transformers のトークナイザと一致するか（ライブラリがあれば）
try:
    from transformers import AutoTokenizer
    hf_tok = AutoTokenizer.from_pretrained(REPO)
    for s in samples + [en, ja]:
        assert hf_tok.encode(s) == encode(s), s
    print("✔ transformers のトークナイザと完全に一致")
except ImportError:
    print("transformers が無いので比較をスキップしました")

## 3. 埋め込み（トークン埋め込み + 学習済み位置埋め込み）

$$
X^{(0)} = W_{te}[\text{ids}] + W_{pe}[0{:}T],\qquad W_{te}\in\mathbb{R}^{|\mathcal{V}|\times d},\ \ W_{pe}\in\mathbb{R}^{n_{\text{ctx}}\times d}
$$

原論文の Transformer と違い、位置埋め込みは正弦波ではなく **学習で得られた行列** です（そのため 1024 トークンより長い入力は扱えません）。また $\sqrt{d}$ のスケーリングはありません。

In [ ]:
text = "The capital of France is"
ids = encode(text)
X0 = wte[ids] + wpe[: len(ids)]
print("tokens:", [decoder[i] for i in ids], " ids:", ids)
print("X0 shape:", X0.shape, "= (T, d)")
print(f"トークン埋め込みの行ノルム（平均）: {np.linalg.norm(wte, axis=1).mean():.3f}")
print(f"位置埋め込みの行ノルム（平均）    : {np.linalg.norm(wpe, axis=1).mean():.3f}")

### 学習された位置埋め込みの構造

学習済みの $W_{pe}$ を調べると、(a) 分散の大きい次元は位置に対して **ゆっくり振動する波のような曲線** になっている、(b) 位置どうしのコサイン類似度が **対角線の近くで高い帯状の構造**（近い位置ほど似ている）を持つ、ということがわかります。これは [transformer.ipynb](transformer.ipynb) で見た正弦波位置エンコーディングの内積行列（Toeplitz 構造）に似た性質を、モデルがデータから自力で獲得したことを示しています。

In [ ]:
wpe_n = wpe / np.linalg.norm(wpe, axis=1, keepdims=True)
cos_pos = wpe_n @ wpe_n.T                                  # (1024, 1024) コサイン類似度
top_dims = np.argsort(-wpe.var(axis=0))[:4]                # 位置方向の分散が大きい次元

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for dim in top_dims:
    axes[0].plot(wpe[:, dim], lw=1, label=f"dim {dim}")
axes[0].set_xlabel("position"); axes[0].set_title("learned W_pe[:, dim] (highest-variance dims)"); axes[0].legend(fontsize=8)
im = axes[1].imshow(cos_pos, cmap="viridis"); axes[1].set_title("cosine similarity of W_pe rows")
axes[1].set_xlabel("position"); axes[1].set_ylabel("position"); plt.colorbar(im, ax=axes[1])
axes[2].plot(np.linalg.norm(wpe, axis=1)); axes[2].set_yscale("log")
axes[2].set_xlabel("position"); axes[2].set_title("||W_pe[pos]||")
plt.tight_layout(); plt.show()
print("位置 100 と他の位置とのコサイン類似度:",
      {k: round(float(cos_pos[100, 100 + k]), 3) for k in [0, 1, 5, 20, 100, 400]})
norms_pe = np.linalg.norm(wpe, axis=1)
print(f"||W_pe[pos]||: 位置 0 = {norms_pe[0]:.2f}, 位置 1〜1022 の平均 = {norms_pe[1:-1].mean():.2f}, 位置 1023 = {norms_pe[-1]:.3f}")

**観察**：右図のように、位置 0 の埋め込みは特に大きく、最後の位置 1023 の埋め込みはほぼ 0 です。後者は因果マスクから説明できると考えられます。位置 1023 の埋め込みは（因果マスクのため）位置 1023 の出力にしか影響しませんが、その出力が予測すべき「1025 番目のトークン」は長さ 1024 の学習系列には存在しないので、損失からの勾配が届きません。勾配がなく weight decay だけがかかり続けた結果、0 に近づいたのでしょう。

### トークン埋め込み空間の近傍

$W_{te}$ の行どうしのコサイン類似度で近傍を調べると、意味や用法の近いトークンが近くに集まっていることがわかります。

In [ ]:
wte_n = wte / np.linalg.norm(wte, axis=1, keepdims=True)


def neighbors(word, k=8):
    ids_w = encode(word)
    assert len(ids_w) == 1, f"{word!r} は 1 トークンではない"
    sims = wte_n @ wte_n[ids_w[0]]
    top = np.argsort(-sims)[1:k + 1]                       # 自分自身を除く
    return [(decode([i]), round(float(sims[i]), 3)) for i in top]


for w in [" Paris", " dog", " happy", " Monday", " 2019"]:
    print(f"{w!r:10s} → {neighbors(w)}")

## 4. Transformer ブロック（Pre-LN）を 1 ステップずつ

GPT-2 のブロック $l$ は、残差ストリーム $x\in\mathbb{R}^{T\times d}$ に 2 つのサブレイヤの出力を **足し込む** 形をしています（Pre-LN）。

$$
\begin{aligned}
\tilde{x} &= \mathrm{LN}_1(x) \\
[\,Q \mid K \mid V\,] &= \tilde{x}\, W_{QKV} + b_{QKV} && (W_{QKV}\in\mathbb{R}^{d\times 3d}) \\
\mathrm{head}_h &= \mathrm{softmax}\!\left(\frac{Q_h K_h^\top}{\sqrt{d_h}} + M\right) V_h && (h = 1,\dots,H,\ \ M:\text{因果マスク}) \\
a &= x + \mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_H)\, W_O + b_O \\
x' &= a + \mathrm{GELU}\big(\mathrm{LN}_2(a)\, W_{fc} + b_{fc}\big)\, W_{proj} + b_{proj}
\end{aligned}
$$

最後のブロックの後に $\mathrm{LN}_f$ をかけ、埋め込み行列の転置でロジットにします（重み共有）：

$$
\text{logits} = \mathrm{LN}_f\big(x^{(L)}\big)\, W_{te}^\top \in \mathbb{R}^{T\times|\mathcal{V}|}
$$

GELU は tanh 近似（`gelu_new`）：$\mathrm{GELU}(x) = \tfrac{1}{2}x\left(1 + \tanh\!\left(\sqrt{2/\pi}\,(x + 0.044715\,x^3)\right)\right)$。

> 第 11 節で行列積の演算回数（FLOPs）を実測するため、行列積はすべて `mm(a, b, tag)` を通します。普段は単なる `a @ b` です。

> **実装上の注意**：`np.sqrt(64)` や `np.sqrt(2 / np.pi)` は NumPy の **float64 スカラー** です。NumPy 2 の型昇格規則では、float32 の配列に float64 スカラーを掛けると結果が float64 に昇格し、以降の行列積のたびに float32 の重み行列全体が float64 に変換（コピー）されて数十倍遅くなります。そこで定数は Python の `float`（`math.sqrt`）で書き、計算全体を float32 のまま保ちます。

In [ ]:
FLOP_COUNTER = None      # None のときは数えない。dict を入れると tag ごとに FLOPs を加算する


def mm(a, b, tag):
    out = a @ b
    if FLOP_COUNTER is not None:
        # 出力の各要素は長さ k の内積（積 k 回 + 和 k 回）→ 2·(出力要素数)·k
        FLOP_COUNTER[tag] = FLOP_COUNTER.get(tag, 0) + 2 * out.size * a.shape[-1]
    return out


def layer_norm(x, g, b, eps=ln_eps):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return g * (x - mu) / np.sqrt(var + eps) + b


GELU_C = math.sqrt(2.0 / math.pi)      # Python の float（np.float64 にすると float32 が float64 に昇格する）
ATTN_SCALE = 1.0 / math.sqrt(d_head)   # 1/√d_h


def gelu(x):
    """tanh 近似の GELU（GPT-2 の gelu_new）"""
    return 0.5 * x * (1.0 + np.tanh(GELU_C * (x + 0.044715 * x ** 3)))


def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    return z - np.log(np.sum(np.exp(z), axis=axis, keepdims=True))


def split_heads(x):
    """(T, d) -> (H, T, d_h)"""
    return x.reshape(x.shape[0], n_heads, d_head).transpose(1, 0, 2)


def merge_heads(x):
    """(H, T, d_h) -> (T, d)"""
    return x.transpose(1, 0, 2).reshape(x.shape[1], n_heads * d_head)


def causal_self_attention(p, x_ln):
    T = x_ln.shape[0]
    qkv = mm(x_ln, p["W_qkv"], "linear") + p["b_qkv"]           # (T, 3d)
    q, k, v = (split_heads(t) for t in np.split(qkv, 3, axis=-1))  # 各 (H, T, d_h)
    S = mm(q, k.transpose(0, 2, 1), "attn") * ATTN_SCALE          # (H, T, T)
    S = np.where(np.tril(np.ones((T, T), dtype=bool)), S, -np.inf)
    A = softmax(S, axis=-1)
    z = merge_heads(mm(A, v, "attn"))                             # (T, d)
    return mm(z, p["W_o"], "linear") + p["b_o"], A, (k, v)


def mlp(p, x_ln):
    h = gelu(mm(x_ln, p["W_fc"], "linear") + p["b_fc"])           # (T, 4d)
    return mm(h, p["W_proj"], "linear") + p["b_proj"]             # (T, d)


def block(p, x):
    attn_out, A, kv = causal_self_attention(p, layer_norm(x, p["ln1_g"], p["ln1_b"]))
    a = x + attn_out
    x_new = a + mlp(p, layer_norm(a, p["ln2_g"], p["ln2_b"]))
    return x_new, A, kv

ブロック 0 を手順どおりに計算して、各段階の形状と値を確認します。`c_attn` の重み $W_{QKV}$ は列方向に $[\,W_Q \mid W_K \mid W_V\,]$ と並んでおり、さらにそれぞれが 64 列ずつ 12 ヘッドに分かれています。

In [ ]:
p0 = blocks[0]
x = X0                                                            # "The capital of France is"
T = x.shape[0]
x_ln = layer_norm(x, p0["ln1_g"], p0["ln1_b"])
print(f"x (残差ストリーム)       : {x.shape}")
print(f"LN_1(x)                  : {x_ln.shape}   各行の平均≈{x_ln.mean(-1).round(3)}（γ,β 適用後なので 0, 1 からずれる）")
qkv = x_ln @ p0["W_qkv"] + p0["b_qkv"]
print(f"[Q|K|V] = LN_1(x) W_QKV  : {qkv.shape}")
Q = x_ln @ p0["W_qkv"][:, :d] + p0["b_qkv"][:d]
assert np.allclose(qkv[:, :d], Q)                                 # 先頭 d 列が Q
Qh = split_heads(Q)
assert np.allclose(Qh[3], Q[:, 3 * d_head:4 * d_head])            # ヘッド 3 = Q の 192〜255 列
print(f"Q をヘッドに分割         : {Qh.shape} = (H, T, d_h)")

attn_out, A0, _ = causal_self_attention(p0, x_ln)
print(f"注意重み A               : {A0.shape} = (H, T, T)")
print(f"注意の出力 (W_O 後)      : {attn_out.shape}")
a = x + attn_out
h_fc = gelu(layer_norm(a, p0["ln2_g"], p0["ln2_b"]) @ p0["W_fc"] + p0["b_fc"])
print(f"MLP 中間層 GELU(·)       : {h_fc.shape} = (T, 4d)")
x1, _, _ = block(p0, x)
print(f"ブロック 0 の出力        : {x1.shape}")
print(f"\n残差ストリームのノルム（位置ごと）: 入力 {np.linalg.norm(x, axis=-1).round(2)}")
print(f"                                  出力 {np.linalg.norm(x1, axis=-1).round(2)}")

ax = heatmap(A0[0], "block 0, head 0 attention", xlabels=[tok_label(i) for i in ids], ylabels=[tok_label(i) for i in ids])
ax.set_xlabel("key"); ax.set_ylabel("query"); plt.show()

## 5. モデル全体の順伝播と `transformers` との一致確認

In [ ]:
def gpt2_forward(ids):
    """ids: (T,) → logits (T, |V|)、各層の残差ストリーム、注意重み、各層の (K, V)"""
    ids = np.asarray(ids)
    T = len(ids)
    assert T <= n_ctx
    x = wte[ids] + wpe[:T]
    resid, attns, kvs = [x], [], []
    for p in blocks:
        x, A, kv = block(p, x)
        resid.append(x)
        attns.append(A)
        kvs.append(kv)
    logits = mm(layer_norm(x, lnf_g, lnf_b), wte.T, "unembed")   # 重み共有: W_out = W_te^T
    return {"logits": logits,              # (T, |V|)
            "resid": resid,                # L+1 個の (T, d)：resid[l] はブロック l の入力
            "attn": np.stack(attns),       # (L, H, T, T)
            "kv": kvs}                     # L 個の (K, V)：各 (H, T, d_h)


t0 = time.perf_counter()
out = gpt2_forward(ids)
print(f"順伝播 {time.perf_counter() - t0:.3f} 秒,  logits shape = {out['logits'].shape}, dtype = {out['logits'].dtype}")
assert out["logits"].dtype == np.float32      # 途中で float64 に昇格していない

Hugging Face `transformers` の `GPT2LMHeadModel` と比較します。`output_hidden_states=True` で得られる `hidden_states[l]`（$l = 0,\dots,11$）はブロック $l$ への入力（＝自前実装の `resid[l]`）で、最後の `hidden_states[12]` だけは $\mathrm{LN}_f$ を適用した後の値です。

In [ ]:
try:
    import torch
    from transformers import AutoModelForCausalLM

    hf_model = AutoModelForCausalLM.from_pretrained(REPO).eval()
    with torch.no_grad():
        hf_out = hf_model(torch.tensor([ids]), output_hidden_states=True)
    hf_logits = hf_out.logits[0].numpy()
    hs = [h[0].numpy() for h in hf_out.hidden_states]
    for l in [0, 1, 6, 11]:
        print(f"hidden_states[{l:2d}] vs resid[{l:2d}]: max|差| = {np.abs(hs[l] - out['resid'][l]).max():.2e}")
    print(f"hidden_states[12] vs LN_f(resid[12]): max|差| = "
          f"{np.abs(hs[12] - layer_norm(out['resid'][12], lnf_g, lnf_b)).max():.2e}")
    diff = np.abs(hf_logits - out["logits"]).max()
    print(f"logits: max|差| = {diff:.2e}  （logits の値の範囲: {out['logits'].min():.1f} 〜 {out['logits'].max():.1f}）")
    assert diff < 1e-2
    print("✔ 自前の NumPy 実装は transformers の GPT-2 と（float32 の丸め誤差の範囲で）一致")
    del hf_model
except ImportError:
    print("torch / transformers が無いので比較をスキップしました")

## 6. 次トークン予測と言語モデルの確率

GPT-2 は **自己回帰言語モデル** で、系列の確率を次のように分解します。

$$
p(x_1, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_{<t}),\qquad p(\cdot \mid x_{\le t}) = \mathrm{softmax}(\text{logits}_t)
$$

位置 $t$ の出力ロジットは **次の位置 $t+1$ のトークン** の予測です。因果マスクのおかげで、1 回の順伝播で全位置の「次トークン予測」が同時に得られます（学習時はこれを使って全位置の損失を一度に計算します）。

学習の目的関数（交差エントロピー）とパープレキシティ：

$$
\mathcal{L} = -\frac{1}{T-1}\sum_{t=1}^{T-1} \log p(x_{t+1}\mid x_{\le t}),\qquad \mathrm{PPL} = \exp(\mathcal{L})
$$

In [ ]:
def next_token_table(text, k=10):
    ids_ = encode(text)
    logits = gpt2_forward(ids_)["logits"][-1].astype(np.float64)
    p = softmax(logits)
    top = np.argsort(-p)[:k]
    print(f"入力: {text!r} → 次トークンの上位 {k} 個")
    for r, i in enumerate(top):
        print(f"  {r + 1:2d}. {decode([i])!r:14s} p = {p[i]:.4f}   logit = {logits[i]:7.2f}")
    print()


next_token_table("The capital of France is")
next_token_table("1, 2, 3, 4, 5,")
next_token_table("import numpy as")

### 各トークンの「驚き」（surprisal）

$-\log_2 p(x_{t+1}\mid x_{\le t})$ [bits] は、モデルにとってそのトークンがどれだけ予想外だったかを表します。文頭のトークンも予測対象にするため、先頭に `<|endoftext|>` を置きます（GPT-2 では文書の区切り記号で、文頭記号としても使われます）。自然な文と、同じ単語を並べ替えた文を比べます。

In [ ]:
def token_nll(text):
    ids_ = [EOT] + encode(text)
    logits = gpt2_forward(ids_)["logits"][:-1].astype(np.float64)     # 位置 t → t+1 の予測
    logp = log_softmax(logits)
    tgt = np.array(ids_[1:])
    return tgt, -logp[np.arange(len(tgt)), tgt]                        # 各トークンの −log p [nats]


natural = "The cat sat on the mat and looked out of the window."
words = natural.rstrip(".").split()
shuffled = " ".join(np.random.default_rng(1).permutation(words)) + "."

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
for ax, s in zip(axes, [natural, shuffled]):
    tgt, nll = token_nll(s)
    ax.bar(range(len(tgt)), nll / np.log(2))
    ax.set_xticks(range(len(tgt)), [tok_label(i) for i in tgt], rotation=45, ha="right")
    ax.set_ylabel("surprisal [bits]")
    ax.set_title(f"{s!r}   mean = {nll.mean():.2f} nats,  PPL = {np.exp(nll.mean()):.1f}")
    print(f"{s!r}: 平均 NLL = {nll.mean():.3f} nats/token, PPL = {np.exp(nll.mean()):.1f}")
plt.tight_layout(); plt.show()
print(f"\n参考: 一様分布なら NLL = log|V| = {np.log(vocab_size):.2f} nats（PPL = {vocab_size}）")

## 7. 注意パターンの観察

学習済みモデルの 144 個のヘッド（12 層 × 12 ヘッド）は、それぞれ特徴的な注意パターンを持ちます。3 種類の現象を数値で探します。

1. **Attention sink**：多くのヘッドが、意味に関係なく **先頭トークン** に大きな注意を向ける。softmax は重みの和が 1 なので、「どこにも注目しない」ことができず、余った重みの捨て場所として先頭トークンが使われる、と解釈されています（Xiao et al., 2023）。
2. **直前トークンヘッド（previous-token head）**：位置 $i$ がほぼ $i-1$ だけを見る。
3. **Induction head**：系列中に `… A B … A` とあるとき、2 回目の `A` の位置から **前回 `A` の直後にあった `B`** に注目し、`B` を次に予測する（Olsson et al., 2022）。文脈内での「コピー」の仕組みで、in-context learning の基礎と考えられています。

各ヘッドについて次のスコアを計算します（$A^{(l,h)}$ は層 $l$・ヘッド $h$ の注意行列）。

$$
\text{sink}_{l,h} = \frac{1}{T-1}\sum_{i=1}^{T-1} A^{(l,h)}_{i,0},\qquad
\text{prev}_{l,h} = \frac{1}{T-1}\sum_{i=1}^{T-1} A^{(l,h)}_{i,i-1}
$$

In [ ]:
sent = "When Mary and John went to the store, John gave a drink to Mary because she was thirsty."
ids7 = [EOT] + encode(sent)
A7 = gpt2_forward(ids7)["attn"]                     # (L, H, T, T)
T7 = len(ids7)
qi = np.arange(1, T7)                               # 位置 0 のクエリは自明（自分しか見られない）なので除く
sink = A7[:, :, qi, 0].mean(-1)                     # (L, H)
prev = A7[:, :, qi, qi - 1].mean(-1)                # (L, H)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
heatmap(sink, "attention to the first token (sink score)", ax=axes[0], fmt="{:.1f}")
heatmap(prev, "attention to the previous token (prev score)", ax=axes[1], fmt="{:.1f}")
for ax in axes:
    ax.set_xlabel("head"); ax.set_ylabel("layer")
plt.tight_layout(); plt.show()

print(f"sink スコアが 0.5 を超えるヘッドの数: {(sink > 0.5).sum()} / {sink.size}")
print("層ごとの sink スコアの平均:", sink.mean(1).round(2))
top_prev = np.dstack(np.unravel_index(np.argsort(-prev, axis=None)[:5], prev.shape))[0]
print("prev スコア上位 5 ヘッド (層.ヘッド):", [f"{l}.{h} ({prev[l, h]:.2f})" for l, h in top_prev])

In [ ]:
labels7 = [tok_label(i) for i in ids7]
l_p, h_p = top_prev[0]
l_s, h_s = np.unravel_index(np.argmax(sink[1:]), sink[1:].shape)
l_s += 1                                                       # 層 1 以降で sink が最大のヘッド
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
for ax, (l, h, name) in zip(axes, [(l_p, h_p, "previous-token head"), (l_s, h_s, "sink head")]):
    im = ax.imshow(A7[l, h], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"{name}: layer {l}, head {h}")
    ax.set_xticks(range(T7), labels7, rotation=90, fontsize=7)
    ax.set_yticks(range(T7), labels7, fontsize=7)
    ax.set_xlabel("key"); ax.set_ylabel("query")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

### Induction head を見つける

ランダムなトークン列 $R$（長さ 50）を 2 回繰り返した入力 `<eot> R R` を使います。1 回目の $R$ はランダムなので予測できませんが、2 回目は「前に見た並びをコピーする」だけで予測できます。

位置 $i$（2 回目の区間）のトークンは、位置 $i-50$ にも出現しています。その **直後の位置 $i - 50 + 1$** に注目するのが induction head です：

$$
\text{induction}_{l,h} = \frac{1}{50}\sum_{i=51}^{100} A^{(l,h)}_{i,\ i-50+1}
$$

In [ ]:
rep_len = 50
rand_tokens = rng.integers(1000, 10000, size=rep_len)
ids_rep = np.concatenate([[EOT], rand_tokens, rand_tokens])        # 長さ 1 + 50 + 50
out_rep = gpt2_forward(ids_rep)
A_rep = out_rep["attn"]
qs = np.arange(rep_len + 1, 2 * rep_len + 1)                        # 2 回目の区間の位置
induction = A_rep[:, :, qs, qs - rep_len + 1].mean(-1)              # (L, H)

# 各位置の次トークン予測の損失
nll_rep = -log_softmax(out_rep["logits"][:-1].astype(np.float64))[np.arange(len(ids_rep) - 1), ids_rep[1:]]

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
heatmap(induction, "induction score", ax=axes[0], fmt="{:.1f}")
axes[0].set_xlabel("head"); axes[0].set_ylabel("layer")
axes[1].plot(np.arange(1, len(ids_rep)), nll_rep, lw=1)
axes[1].axvline(rep_len + 0.5, color="gray", ls="--", label="start of the 2nd copy")
axes[1].set_xlabel("position of the predicted token"); axes[1].set_ylabel("NLL [nats]")
axes[1].set_title("per-token loss on  <eot> R R"); axes[1].legend()
plt.tight_layout(); plt.show()

top_ind = np.dstack(np.unravel_index(np.argsort(-induction, axis=None)[:6], induction.shape))[0]
print("induction スコア上位 6 ヘッド (層.ヘッド):", [f"{l}.{h} ({induction[l, h]:.2f})" for l, h in top_ind])
print(f"1 回目の区間の平均損失: {nll_rep[:rep_len].mean():.2f} nats  →  2 回目: {nll_rep[rep_len:].mean():.2f} nats")

In [ ]:
l_i, h_i = top_ind[0]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (l, h, name) in zip(axes, [(l_i, h_i, "induction head"), (top_prev[0][0], top_prev[0][1], "previous-token head")]):
    im = ax.imshow(A_rep[l, h], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"{name}: layer {l}, head {h}  on  <eot> R R")
    ax.set_xlabel("key position"); ax.set_ylabel("query position")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

**観察**：
- 2 回目の区間に入ると損失が急落します。モデルは **重みを一切更新せずに**、文脈の中の情報だけで並びをコピーできています（in-context learning の最も単純な例）。
- induction スコアが高いのは層 5〜7 のヘッド（5.1, 5.5, 6.9, 7.2, 7.10 など）で、GPT-2 small の induction head として知られているものと一致します。その注意行列には、対角線から 49 だけ左にずれた位置に **斜めの線** が現れます（「前回の出現位置の次」を見ている）。
- 直前トークンヘッド（4.11 や層 2〜3 のヘッド）は induction head より **浅い層** にあります。Olsson et al. (2022) は、浅い層の直前トークンヘッドが各位置に「直前のトークンが何だったか」を書き込み、深い層の induction head がそれをキーとして使う、という 2 段階の回路で induction が実現されると説明しています。

## 8. Logit lens：層を追って予測が形成される様子

Pre-LN の GPT-2 では、残差ストリーム $x^{(l)}$ はどの層でも同じ $d$ 次元空間にあり、各ブロックはそこに少しずつ情報を足し込みます。そこで **途中の層の残差ストリームに最終層と同じ $\mathrm{LN}_f$ と $W_{te}^\top$ をかけて** 語彙上の分布を読むと、予測が層を追ってどう形成されるかが見えます（logit lens, nostalgebraist 2020）。

$$
\text{logits}^{(l)} = \mathrm{LN}_f\big(x^{(l)}\big)\, W_{te}^\top,\qquad l = 0, 1, \dots, L
$$

$l = L$ のときは通常の出力と一致します。

In [ ]:
prompt8 = "The capital of France is"
ids8 = encode(prompt8)
out8 = gpt2_forward(ids8)
lens_logits = np.stack([layer_norm(r, lnf_g, lnf_b) @ wte.T for r in out8["resid"]])   # (L+1, T, |V|)
assert np.allclose(lens_logits[-1], out8["logits"], atol=1e-4)
probs8 = softmax(lens_logits.astype(np.float64), axis=-1)

target = encode(" Paris")[0]
print(f"入力 {prompt8!r} の最後の位置で、各層の残差ストリームが予測する次トークン:\n")
print(f"{'層':>3} | {'top-1':14s} {'p(top-1)':>8} | {'p(Paris)':>9} {'順位':>6}")
p_target, rank_target = [], []
for l in range(n_layers + 1):
    p = probs8[l, -1]
    top1 = int(np.argmax(p))
    rank = int((p > p[target]).sum()) + 1
    p_target.append(p[target]); rank_target.append(rank)
    print(f"{l:>3} | {decode([top1])!r:14s} {p[top1]:8.4f} | {p[target]:9.5f} {rank:>6}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(p_target, "o-"); axes[0].set_title("p(' Paris') at the last position"); axes[0].set_xlabel("layer")
axes[1].plot(rank_target, "o-"); axes[1].set_yscale("log"); axes[1].invert_yaxis()
axes[1].set_title("rank of ' Paris' (1 = top)"); axes[1].set_xlabel("layer")
plt.tight_layout(); plt.show()

**観察**：`' Paris'` は浅い層では数万位、層 9 でもまだ 50 位程度ですが、層 10 で一気に 2 位まで浮上します。つまり「フランスの首都」という知識は浅い層ではなく **深い層で残差ストリームに書き込まれている** ことがわかります。一方、最終層では確率が `' the'` などの機能語に移り、`' Paris'` は 5 位に下がります（"The capital of France is the city of Paris" のような続きも十分ありうるため）。GPT-2 small では、正解の知識が途中の層で現れても最終出力の 1 位になるとは限りません。

文全体について、各層・各位置での top-1 予測を表示します（セル内の文字列は「その位置の **次** に来ると予測したトークン」、色はその確率）。

In [ ]:
sent8 = "The Eiffel Tower is located in the city of"
ids8b = encode(sent8)
out8b = gpt2_forward(ids8b)
P8 = softmax(np.stack([layer_norm(r, lnf_g, lnf_b) @ wte.T for r in out8b["resid"]]).astype(np.float64), axis=-1)
top1 = P8.argmax(-1)                                    # (L+1, T)
ptop = P8.max(-1)

fig, ax = plt.subplots(figsize=(1.05 * len(ids8b) + 2, 6.5))
im = ax.imshow(ptop, cmap="Blues", vmin=0, vmax=1, aspect="auto")
for l in range(n_layers + 1):
    for t in range(len(ids8b)):
        ax.text(t, l, tok_label(top1[l, t]), ha="center", va="center", fontsize=7,
                color="w" if ptop[l, t] > 0.5 else "k")
ax.set_xticks(range(len(ids8b)), [tok_label(i) for i in ids8b], rotation=45, ha="right")
ax.set_yticks(range(n_layers + 1)); ax.set_ylabel("layer (0 = embeddings)"); ax.invert_yaxis()
ax.set_title("logit lens: top-1 next-token prediction at each layer / position")
plt.colorbar(im, ax=ax, fraction=0.03, label="p(top-1)")
plt.tight_layout(); plt.show()

**観察**：最後の位置（`of` の次）では、中間層でまず `England` や `East`、続いて `Rome` や `London` といった **地名・都市名** が現れ、層 11 以降で正解の `Paris` に落ち着きます。「ここには都市名が来る」という大まかな予測が先にでき、深い層で具体的な答えに絞り込まれていく様子が見えます。また `Eiffel` の途中（`E` → `iff` → `el`）では、浅い層から `el` → `Tower` のように単語の続きを高い確率で当てています。

## 9. 残差ストリームのノルム

[transformer.ipynb](transformer.ipynb) の第 10 節では、ランダム重みの Pre-LN で残差ストリームのノルムが層とともに増えることを見ました。学習済みの GPT-2 でも確認します。さらに学習済みモデルでは、**中間層で先頭トークンの残差ストリームだけが桁違いに大きくなる** という現象（massive activations, Sun et al., 2024）が見られます。第 7 節の attention sink と関係していると考えられています。

In [ ]:
sent9 = "The quick brown fox jumps over the lazy dog. It was a sunny day in the park."
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, (name, ids9) in zip(axes, [("with <eot> at position 0", [EOT] + encode(sent9)),
                                   ("without <eot>", encode(sent9))]):
    norms9 = np.stack([np.linalg.norm(r, axis=-1) for r in gpt2_forward(ids9)["resid"]])   # (L+1, T)
    ax.plot(norms9[:, 0], "o-", label=f"position 0 ({tok_label(ids9[0])})")
    ax.plot(norms9[:, 1:].mean(1), "s-", label="mean of positions 1..T-1")
    ax.set_yscale("log"); ax.set_xlabel("layer (input of block l; 12 = after the last block)"); ax.set_ylabel("||x||")
    ax.set_title(name); ax.legend()
    print(f"[{name}]")
    print("  層          :", " ".join(f"{l:6d}" for l in range(n_layers + 1)))
    print("  位置 0      :", " ".join(f"{v:6.0f}" for v in norms9[:, 0]))
    print("  他の位置平均:", " ".join(f"{v:6.0f}" for v in norms9[:, 1:].mean(1)))
plt.tight_layout(); plt.show()

**観察**：
- 先頭以外の位置では、ノルムは層とともに徐々に大きくなります（Pre-LN では各ブロックの出力が残差ストリームに足され続けるため）。
- 先頭の位置だけは、最初の 3 ブロックで一気に約 3000 まで大きくなり、層 3〜11 ではほかの位置の十数倍〜数十倍の大きさを保ちます。そして最後のブロックで急に小さくなります。
- 先頭が `<eot>` でも `The` でも同じことが起きるので、トークンの種類ではなく **「先頭の位置であること」** によって起きる現象です。どの位置からも必ず見える先頭トークンを、attention sink（第 7 節）の置き場として使っていると考えられています。

## 10. 生成：KV キャッシュとデコード戦略

### 10.1 KV キャッシュ（prefill と decode）

実際の LLM 推論は 2 段階に分かれます。

- **prefill**：プロンプト全体を 1 回の順伝播で処理し、各層の $K, V$（形状 $(H, T, d_h)$）を保存する。
- **decode**：新しいトークン 1 個だけを入力し、その $q_t, k_t, v_t$ を計算して $K, V$ に追記、$q_t$ でキャッシュ全体に注意する。

$$
K_{1:t} = \begin{pmatrix} K_{1:t-1} \\ k_t \end{pmatrix},\quad
V_{1:t} = \begin{pmatrix} V_{1:t-1} \\ v_t \end{pmatrix},\quad
z_t = \mathrm{softmax}\!\left(\frac{q_t K_{1:t}^\top}{\sqrt{d_h}}\right) V_{1:t}
$$

In [ ]:
def gpt2_step(token_id, pos, cache):
    """decode ステップ：1 トークンだけ処理して各層の (K, V) に追記し、次トークンのロジットを返す"""
    x = wte[[token_id]] + wpe[[pos]]                                # (1, d)
    for l, p in enumerate(blocks):
        x_ln = layer_norm(x, p["ln1_g"], p["ln1_b"])
        qkv = x_ln @ p["W_qkv"] + p["b_qkv"]
        q, k, v = (split_heads(t) for t in np.split(qkv, 3, axis=-1))   # 各 (H, 1, d_h)
        K = np.concatenate([cache[l][0], k], axis=1)                # (H, t, d_h)
        V = np.concatenate([cache[l][1], v], axis=1)
        cache[l] = (K, V)
        A = softmax(q @ K.transpose(0, 2, 1) * ATTN_SCALE)          # (H, 1, t)：キャッシュには過去しかないのでマスク不要
        x = x + merge_heads(A @ V) @ p["W_o"] + p["b_o"]
        x = x + mlp(p, layer_norm(x, p["ln2_g"], p["ln2_b"]))
    return (layer_norm(x, lnf_g, lnf_b) @ wte.T)[0]                 # (|V|,)


def filter_logits(logits, temperature=1.0, top_k=None, top_p=None):
    """温度・top-k・top-p を適用したロジット（除外したトークンは -inf）"""
    z = logits.astype(np.float64) / temperature
    if top_k is not None:
        kth = np.sort(z)[-top_k]
        z = np.where(z < kth, -np.inf, z)
    if top_p is not None:
        order = np.argsort(-z)
        cum = np.cumsum(softmax(z[order]))
        n_keep = int(np.searchsorted(cum, top_p)) + 1        # 累積確率が top_p 以上になる最小の個数
        z = z.copy()
        z[order[n_keep:]] = -np.inf
    return z


def sample_next(logits, rng=None, greedy=False, temperature=1.0, top_k=None, top_p=None):
    if greedy:
        return int(np.argmax(logits))
    p = softmax(filter_logits(logits, temperature, top_k, top_p))
    return int(rng.choice(len(p), p=p / p.sum()))


def generate(prompt_ids, n_new, rng=None, greedy=False, temperature=1.0, top_k=None, top_p=None):
    out = gpt2_forward(prompt_ids)                       # prefill
    cache = list(out["kv"])
    logits = out["logits"][-1]
    ids_, step_logits = list(prompt_ids), []
    for _ in range(n_new):
        step_logits.append(logits)
        nxt = sample_next(logits, rng, greedy, temperature, top_k, top_p)
        ids_.append(nxt)
        if len(ids_) == n_ctx:
            break
        logits = gpt2_step(nxt, len(ids_) - 1, cache)    # decode
    return ids_, step_logits

In [ ]:
prompt_ids = encode("The capital of France is")
gen_ids, step_logits = generate(prompt_ids, 15, greedy=True)
print("greedy 生成:", repr(decode(gen_ids)))

# KV キャッシュで得たロジットと、生成後の系列全体を一度に順伝播したロジットを比較
full_logits = gpt2_forward(gen_ids)["logits"]
P = len(prompt_ids)
diffs = [np.abs(step_logits[j] - full_logits[P - 1 + j]).max() for j in range(len(step_logits))]
print(f"各ステップの max|logits 差|: 最大 {max(diffs):.2e}")
assert max(diffs) < 1e-2
print("✔ KV キャッシュを使った逐次計算は、全体の再計算と一致")

# 速度の比較（素朴な実装：毎ステップ系列全体を順伝播）
n_new = 40
t0 = time.perf_counter()
seq = list(prompt_ids)
for _ in range(n_new):
    seq.append(int(np.argmax(gpt2_forward(seq)["logits"][-1])))
t_naive = time.perf_counter() - t0
t0 = time.perf_counter()
seq_cached, _ = generate(prompt_ids, n_new, greedy=True)
t_cached = time.perf_counter() - t0
assert seq == seq_cached
print(f"\n{n_new} トークン生成: 毎回再計算 {t_naive:.2f} 秒 / KV キャッシュ {t_cached:.2f} 秒（同じ結果）")

### 10.2 デコード戦略

次トークン分布 $p = \mathrm{softmax}(z)$ からどう選ぶかで、生成文の性質が大きく変わります。

| 戦略 | 定義 |
|---|---|
| greedy | $w = \arg\max_w p_w$ |
| 温度 $\tau$ | $p_w \propto \exp(z_w/\tau)$（$\tau<1$ で鋭く、$\tau>1$ で平らに） |
| top-$k$ | 確率上位 $k$ 個だけを残して再正規化 |
| top-$p$（nucleus） | 確率の大きい順に足して累積が $p$ 以上になる **最小の集合** $V^{(p)}$ だけを残す：$\sum_{w\in V^{(p)}} p_w \ge p$（Holtzman et al., 2020） |

top-$k$ は残す個数が固定ですが、top-$p$ は **分布の形に応じて残す個数が変わる**（自信があるときは少なく、迷っているときは多く）のが特徴です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, prompt in zip(axes, ["1, 2, 3, 4, 5,", "Once upon a time, there was a"]):
    logits = gpt2_forward(encode(prompt))["logits"][-1]
    p = softmax(logits.astype(np.float64))
    order = np.argsort(-p)[:30]
    kept_k = np.isfinite(filter_logits(logits, top_k=10))
    kept_p = np.isfinite(filter_logits(logits, top_p=0.9))
    colors = ["tab:green" if kept_p[i] else "tab:gray" for i in order]
    ax.bar(range(30), p[order], color=colors)
    ax.set_xticks(range(30), [tok_label(i) for i in order], rotation=90, fontsize=8)
    ax.set_title(f"{prompt!r}\ntop-p=0.9 keeps {kept_p.sum()} tokens (green), top-k=10 keeps {kept_k.sum()}")
    print(f"{prompt!r}: top-p=0.9 で残るトークン数 = {kept_p.sum()},  エントロピー = {-(p * np.log(p + 1e-30)).sum():.2f} nats")
plt.tight_layout(); plt.show()

In [ ]:
def distinct_n(ids_, n=3):
    """生成部分の n-gram のうち異なるものの割合（小さいほど繰り返しが多い）"""
    grams = [tuple(ids_[i:i + n]) for i in range(len(ids_) - n + 1)]
    return len(set(grams)) / max(1, len(grams))


prompt = ("In a shocking finding, scientists discovered a herd of unicorns living in a remote, "
          "previously unexplored valley in the Andes Mountains.")
pids = encode(prompt)
n_gen = 100
settings = [("greedy", dict(greedy=True)),
            ("temperature 0.5", dict(temperature=0.5)),
            ("temperature 1.0", dict(temperature=1.0)),
            ("top-k 40", dict(top_k=40)),
            ("top-p 0.9", dict(top_p=0.9))]
for name, kw in settings:
    seeds = [0] if kw.get("greedy") else range(5)          # greedy は決定的なので 1 回だけ
    conts = [generate(pids, n_gen, rng=np.random.default_rng(s), **kw)[0][len(pids):] for s in seeds]
    d3 = np.mean([distinct_n(c) for c in conts])
    print(f"=== {name}   distinct-3 = {d3:.2f}（{len(conts)} 回の平均） ===")
    print(decode(conts[0]).strip()[:600], "\n")

**観察**：
- greedy では、`… were able to survive in the wild for up to 10 years, and that they were able to survive in the wild for up to …` のように **同じ句が繰り返され** やすく、一度ループに入ると抜け出せません（決定的なので、同じ状態からは必ず同じ選択をする）。第 10.1 節の `The capital of France is the capital of the French Republic, and the capital of the French Republic is` も同じ現象です。Holtzman et al. (2020) はこれを "neural text degeneration" と呼びました。
- 温度を下げると greedy に近づき、繰り返しが増えます（distinct-3 が下がる）。
- 温度 1.0 でそのままサンプリングすると、低確率の「裾」から変なトークンを引いて文が崩れることがあります。top-$k$ / top-$p$ は裾を切り落としてこれを防ぐ工夫です。

## 11. スケーリング則

GPT-2 の論文は、パラメータ数を 117M → 1.5B と増やすほど言語モデルの性能が上がることを示しました。Kaplan et al. (2020) はこれを定量化し、損失が **パラメータ数 $N$・データ量 $D$（トークン数）・計算量 $C$（FLOPs）の冪乗則** に従うことを示しました。さらに Hoffmann et al. (2022, Chinchilla) は、計算量を固定したときに $N$ と $D$ をどう配分すべきかを見直しました。

このノートでは学習を行わないので、ここでは

1. スケーリング則の基本量 $N$ と $C \approx 6ND$ を、**実際の GPT-2 の重みと自前実装の FLOPs の実測** で確かめる。
2. Kaplan の冪乗則の式を計算して性質を確かめる。
3. Chinchilla の式から **計算最適な $N$ と $D$ を導出** し、数値的に検証する。

という順で進めます。

### 11.1 パラメータ数 $N \approx 12 L d^2$

Kaplan et al. は **埋め込みを除いた** パラメータ数を $N$ とします（埋め込みを含めるとスケーリングがきれいな冪乗則にならないため）。第 1 節の表から、1 ブロックの重み行列は

$$
\underbrace{3d^2}_{W_{QKV}} + \underbrace{d^2}_{W_O} + \underbrace{4d^2}_{W_{fc}} + \underbrace{4d^2}_{W_{proj}} = 12d^2
\quad\Rightarrow\quad N \approx 12\, L\, d^2
$$

GPT-2 の 4 つのサイズ（論文 Table 2 の層数と次元）について計算します。

In [ ]:
gpt2_family = {"small": (12, 768), "medium": (24, 1024), "large": (36, 1280), "xl": (48, 1600)}


def count_gpt2(L_, d_, V_=vocab_size, ctx=n_ctx):
    nonemb = L_ * (12 * d_ * d_ + 13 * d_) + 2 * d_        # ブロック + ln_f
    emb = V_ * d_ + ctx * d_
    return nonemb, emb


print(f"{'model':7s} {'L':>3} {'d':>5} | {'N (非埋め込み)':>15} {'12·L·d²':>15} | {'埋め込み':>12} {'合計':>15}")
for name, (L_, d_) in gpt2_family.items():
    nonemb, emb = count_gpt2(L_, d_)
    print(f"{name:7s} {L_:>3} {d_:>5} | {nonemb:>15,} {12 * L_ * d_ * d_:>15,} | {emb:>12,} {nonemb + emb:>15,}")
assert sum(count_gpt2(n_layers, d_model)) == n_total       # small は実際の重みの数と一致
print("\n✔ small の合計は、読み込んだ実際の重みの要素数と一致")
print("  （GPT-2 論文の表では 117M / 345M / 762M / 1542M と書かれているが、実際の重みの数は上の通り）")

### 11.2 計算量 $C \approx 6ND$ を実測で確かめる

**順伝播**：長さ $k$ の内積は積 $k$ 回と和 $k$ 回なので、$(m\times k)$ と $(k\times n)$ の行列積は $2mkn$ FLOPs です。重み行列 $W\in\mathbb{R}^{k\times n}$ に 1 トークン（$1\times k$）を通すと $2kn$ FLOPs、つまり **パラメータ 1 個あたり 2 FLOPs** です。よって 1 トークンの順伝播は

$$
C_{\text{forward}} \approx 2N + \underbrace{2\, L\, n_{\text{ctx}}\, d}_{\text{注意 }QK^\top, AV} \quad(\text{Kaplan et al. の Table 1})
$$

注意の項は、各クエリが平均 $n_{\text{ctx}}/2$ 個のキー（因果マスク）と $QK^\top$ と $AV$ の 2 回の積をとることに対応します。本実装は $T\times T$ 全体を計算してからマスクするので $4LTd$ になります（因果マスクの分だけ 2 倍の無駄がある）。出力層（$2d|\mathcal{V}|$）は埋め込みと同様に Kaplan らの $C$ には含めません。

まず、`mm()` の FLOP カウンタで実測した値がこの式と **完全に一致** することを確かめます。

In [ ]:
T_meas = 256
FLOP_COUNTER = {}
_ = gpt2_forward(rng.integers(0, vocab_size, size=T_meas))
measured, FLOP_COUNTER = FLOP_COUNTER, None

N_mat = n_layers * 12 * d_model ** 2                       # 重み行列の要素数（バイアス・LN を除く）
expected = {"linear": 2 * N_mat * T_meas,                  # 2N / トークン
            "attn": 4 * n_layers * T_meas * T_meas * d_model,   # T×T を全部計算
            "unembed": 2 * d_model * vocab_size * T_meas}
for k in expected:
    print(f"{k:8s}: 実測 {measured[k]:>16,}  式 {expected[k]:>16,}  → 1 トークンあたり {measured[k] / T_meas / 1e6:8.1f} MFLOPs")
    assert measured[k] == expected[k]
print("✔ 行列積の FLOPs の実測値は式と一致")
print(f"\n2N = {2 * N_nonemb / 1e6:.1f} MFLOPs / トークン（N = 埋め込みを除くパラメータ数）")

**逆伝播 ≈ 順伝播の 2 倍**：線形層 $Y = XW$（$X\in\mathbb{R}^{m\times k}, W\in\mathbb{R}^{k\times n}$）の逆伝播では、入力への勾配 $\partial\mathcal{L}/\partial X = (\partial\mathcal{L}/\partial Y)\,W^\top$ と重みへの勾配 $\partial\mathcal{L}/\partial W = X^\top(\partial\mathcal{L}/\partial Y)$ の **2 回の同じ大きさの行列積** が必要です。したがって学習 1 トークンあたり $2N + 4N = 6N$、$D$ トークンで

$$
C \approx 6ND
$$

1 つの線形層で、勾配の正しさ（数値微分との比較）と FLOPs の比を確かめます。

In [ ]:
m_, k_, n_ = 16, 64, 32
Xl = rng.standard_normal((m_, k_))
Wl = rng.standard_normal((k_, n_))
G = rng.standard_normal((m_, n_))            # 損失 L = Σ (XW) ⊙ G とすると ∂L/∂Y = G

FLOP_COUNTER = {}
Y = mm(Xl, Wl, "forward")
dX = mm(G, Wl.T, "backward")                 # ∂L/∂X = G W^T
dW = mm(Xl.T, G, "backward")                 # ∂L/∂W = X^T G
fl, FLOP_COUNTER = FLOP_COUNTER, None

eps = 1e-6
E = np.zeros_like(Wl); E[3, 5] = eps
num = (np.sum((Xl @ (Wl + E)) * G) - np.sum((Xl @ (Wl - E)) * G)) / (2 * eps)
assert np.isclose(num, dW[3, 5])
print(f"∂L/∂W[3,5]: 解析 {dW[3, 5]:.6f} / 数値微分 {num:.6f}")
print(f"FLOPs: 順伝播 {fl['forward']:,} / 逆伝播 {fl['backward']:,}  → 比 {fl['backward'] / fl['forward']:.1f}")

In [ ]:
PF_DAY = 1e15 * 86400          # 1 PF-day = 10^15 FLOP/s で 1 日 ≈ 8.64e19 FLOP
print(f"{'model':7s} | {'2N':>9} {'注意(因果)':>10} {'出力層':>9} | 注意/2N  出力層/2N   (MFLOPs/トークン, n_ctx=1024)")
for name, (L_, d_) in gpt2_family.items():
    N_, _ = count_gpt2(L_, d_)
    att = 2 * L_ * n_ctx * d_
    une = 2 * d_ * vocab_size
    print(f"{name:7s} | {2 * N_ / 1e6:9.1f} {att / 1e6:10.1f} {une / 1e6:9.1f} | {att / (2 * N_):6.1%}  {une / (2 * N_):8.1%}")

D_example = 10e9
C_example = 6 * N_nonemb * D_example
print(f"\n例: GPT-2 small (N={N_nonemb / 1e6:.0f}M) を D=10B トークンで学習 → C ≈ 6ND = {C_example:.2e} FLOP"
      f" = {C_example / PF_DAY:.3f} PF-days")

**観察**：モデルが大きくなるほど注意と出力層の割合は小さくなり、$C_{\text{forward}}\approx 2N$ がよい近似になります。GPT-2 small のような小さいモデルでは、出力層（語彙 50257 への射影）の計算が無視できない大きさです。

### 11.3 Kaplan et al. (2020) の冪乗則

WebText2 上のテスト損失 $L$（nats/トークン）は、他の 2 つの量がボトルネックにならない条件で

$$
L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N},\qquad
L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D},\qquad
L(C_{\min}) = \left(\frac{C_c^{\min}}{C_{\min}}\right)^{\alpha_C^{\min}}
$$

| | 指数 | 定数 |
|---|---|---|
| $N$（埋め込みを除くパラメータ数） | $\alpha_N \approx 0.076$ | $N_c \approx 8.8\times 10^{13}$ |
| $D$（トークン数） | $\alpha_D \approx 0.095$ | $D_c \approx 5.4\times 10^{13}$ |
| $C_{\min}$（PF-days） | $\alpha_C^{\min} \approx 0.050$ | $C_c^{\min} \approx 3.1\times 10^{8}$ |

冪乗則は両対数グラフで **直線** になります：$\log L = \alpha_N(\log N_c - \log N)$。また $N$ を $k$ 倍すると損失は $k^{-\alpha_N}$ 倍になり、これは $N$ の大きさによりません。

（注：これらの値は WebText2 と GPT-2 のトークナイザでの損失についてのもので、第 6 節で計算した損失とは直接比べられません。下の GPT-2 の点も「この式に $N$ を代入した値」であり、GPT-2 の実測損失ではありません。）

In [ ]:
alpha_N, N_c = 0.076, 8.8e13
alpha_D, D_c = 0.095, 5.4e13
alpha_C, C_c = 0.050, 3.1e8
L_N = lambda N: (N_c / N) ** alpha_N
L_D = lambda D: (D_c / D) ** alpha_D
L_C = lambda C: (C_c / C) ** alpha_C          # C は PF-days

for k in [2, 10, 100]:
    print(f"N を {k:>3} 倍 → 損失は {k ** -alpha_N:.4f} 倍（N=1e8 で確認: {L_N(k * 1e8) / L_N(1e8):.4f}, N=1e10: {L_N(k * 1e10) / L_N(1e10):.4f}）")
print(f"損失を半分にするには N を 2^(1/α_N) = {2 ** (1 / alpha_N):,.0f} 倍にする必要がある\n")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
Ns = np.logspace(5, 12, 100)
axes[0].loglog(Ns, L_N(Ns))
for name, (L_, d_) in gpt2_family.items():
    N_, _ = count_gpt2(L_, d_)
    axes[0].plot(N_, L_N(N_), "o"); axes[0].annotate(f"GPT-2 {name}", (N_, L_N(N_)), fontsize=8)
    print(f"GPT-2 {name:6s}: N = {N_:.3g} → L(N) = {L_N(N_):.3f} nats")
axes[0].set_xlabel("N (non-embedding parameters)"); axes[0].set_ylabel("loss [nats]"); axes[0].set_title("L(N)")
Ds = np.logspace(7, 13, 100)
axes[1].loglog(Ds, L_D(Ds)); axes[1].set_xlabel("D (tokens)"); axes[1].set_title("L(D)")
Cs = np.logspace(-8, 4, 100)
axes[2].loglog(Cs, L_C(Cs)); axes[2].set_xlabel("C_min (PF-days)"); axes[2].set_title("L(C_min)")
for ax in axes:
    ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

#### $N$ と $D$ を同時に考える

Kaplan et al. は $N$ と $D$ の両方を含む式も提案しています（定数は論文 Table 2 の $L(N,D)$ へのフィット：$\alpha_N=0.076,\ \alpha_D=0.103,\ N_c=6.4\times10^{13},\ D_c=1.8\times10^{13}$）。

$$
L(N, D) = \left[\left(\frac{N_c}{N}\right)^{\alpha_N/\alpha_D} + \frac{D_c}{D}\right]^{\alpha_D}
$$

- $D\to\infty$ で $L(N)$ に、$N\to\infty$ で $L(D)$ に一致します。
- 有限の $D$ では、$N$ を大きくしても損失は $L(D)$ 以下には下がりません（データが足りず **過学習** する領域）。
- 過学習の影響が小さいのは、括弧内の第 2 項が第 1 項より十分小さいとき：$\dfrac{D_c}{D} \ll \left(\dfrac{N_c}{N}\right)^{\alpha_N/\alpha_D} \iff D \gg D_c\left(\dfrac{N}{N_c}\right)^{\alpha_N/\alpha_D}$。つまり **必要なデータ量は $N^{\alpha_N/\alpha_D} \approx N^{0.74}$ に比例** して増えます（モデルを 8 倍にしてもデータは約 5 倍でよい）。

In [ ]:
aN2, aD2, Nc2, Dc2 = 0.076, 0.103, 6.4e13, 1.8e13
L_ND = lambda N, D: ((Nc2 / N) ** (aN2 / aD2) + Dc2 / D) ** aD2

N_big, D_big = 1e9, 1e9
print(f"D→∞ の極限: L(N=1e9, D=1e20) = {L_ND(N_big, 1e20):.4f}  vs  (N_c/N)^α_N = {(Nc2 / N_big) ** aN2:.4f}")
print(f"N→∞ の極限: L(N=1e20, D=1e9) = {L_ND(1e20, D_big):.4f}  vs  (D_c/D)^α_D = {(Dc2 / D_big) ** aD2:.4f}")
print(f"必要なデータ量の指数 α_N/α_D = {aN2 / aD2:.3f}  → N を 8 倍にするとデータは {8 ** (aN2 / aD2):.2f} 倍")

fig, ax = plt.subplots(figsize=(7, 4.5))
Ds = np.logspace(7, 12, 200)
for name, (L_, d_) in gpt2_family.items():
    N_, _ = count_gpt2(L_, d_)
    line, = ax.loglog(Ds, L_ND(N_, Ds), label=f"N = GPT-2 {name}")
    ax.axhline((Nc2 / N_) ** aN2, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("D (tokens)"); ax.set_ylabel("loss [nats]")
ax.set_title("Kaplan L(N, D): dotted = L(N) limit as D → ∞")
ax.legend(); ax.grid(alpha=0.3, which="both"); plt.show()

Kaplan et al. は、計算量 $C$ を最適に使うには $N_{\text{opt}}\propto C^{0.73}$、つまり **計算量の大部分をモデルの大型化に回し、データはあまり増やさない** のがよいと結論しました（$C = 6ND$ より $D\propto C^{0.27}$）。GPT-3（175B パラメータ・約 300B トークン）はこの考え方の延長線上にあります。

### 11.4 Hoffmann et al. (2022, Chinchilla) の計算最適配分

Hoffmann et al. は、学習率スケジュールなどを各設定で調整し直して大量のモデルを学習し、次の形の式をフィットしました。

$$
L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}
$$

$E$ はデータそのもののエントロピー（どんなに大きなモデルでも下げられない損失）、第 2 項はモデルが有限であることによる損失、第 3 項はデータが有限であることによる損失です。

**計算最適配分の導出**：制約 $C = 6ND$ のもとで $L$ を最小化します。$D = C/(6N)$ を代入して $N$ で微分すると

$$
\frac{dL}{dN} = -\alpha A N^{-\alpha-1} + \beta B\left(\frac{C}{6}\right)^{-\beta} N^{\beta-1} = 0
\;\Longrightarrow\;
N^{\alpha+\beta} = \frac{\alpha A}{\beta B}\left(\frac{C}{6}\right)^{\beta}
$$

$$
\therefore\quad
N_{\text{opt}} = G\left(\frac{C}{6}\right)^{a},\qquad
D_{\text{opt}} = G^{-1}\left(\frac{C}{6}\right)^{b},\qquad
a = \frac{\beta}{\alpha+\beta},\quad b = \frac{\alpha}{\alpha+\beta},\quad G = \left(\frac{\alpha A}{\beta B}\right)^{\frac{1}{\alpha+\beta}}
$$

$a + b = 1$ で、$\alpha\approx\beta$ なら $a\approx b\approx 0.5$：**計算量を増やすときはモデルとデータを同じ割合で増やす** のが最適、という結論になります。Hoffmann et al. はこれに従って、Gopher（280B パラメータ）と同程度の計算量で **70B パラメータ・1.4T トークン**（約 20 トークン/パラメータ）の Chinchilla を学習し、Gopher を上回る性能を得ました。

定数については 2 組を比べます。
- **Hoffmann et al. の値**：$E=1.6934,\ A=406.4,\ B=410.7,\ \alpha=0.3392,\ \beta=0.2849$（論文本文の丸めた値は $E=1.69,\ \alpha=0.34,\ \beta=0.28$。ここでは Besiroglu et al. が Table 1 に記載した桁数の値を使う）
- **Besiroglu et al. (2024) の再推定**：$E=1.8172,\ A=482.01,\ B=2085.43,\ \alpha=0.3478,\ \beta=0.3658$。Hoffmann et al. の値は同じ論文のほかの 2 つの推定方法や Chinchilla 自体の学習設定と食い違っていると指摘し、データを再分析して得た値です。

In [ ]:
fits = {
    "Hoffmann et al. (2022)":  dict(E=1.6934, A=406.4,  B=410.7,   alpha=0.3392, beta=0.2849),
    "Besiroglu et al. (2024)": dict(E=1.8172, A=482.01, B=2085.43, alpha=0.3478, beta=0.3658),
}


def chin_loss(N, D, E, A, B, alpha, beta):
    return E + A / N ** alpha + B / D ** beta


def chin_opt(C, E, A, B, alpha, beta):
    """閉形式の計算最適解 (N_opt, D_opt)"""
    a, b = beta / (alpha + beta), alpha / (alpha + beta)
    G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
    return G * (C / 6) ** a, (C / 6) ** b / G


C_chin = 6 * 70e9 * 1.4e12             # Chinchilla 自体の学習計算量
print(f"C = 6 × 70B × 1.4T = {C_chin:.3e} FLOP（Chinchilla の学習計算量）\n")
for name, f in fits.items():
    a = f["beta"] / (f["alpha"] + f["beta"])
    N_opt, D_opt = chin_opt(C_chin, **f)
    # 数値的な検証：等計算量曲線 D = C/(6N) の上で N を細かく振って最小値を探す
    Ngrid = np.logspace(9, 12, 200001)
    Lgrid = chin_loss(Ngrid, C_chin / (6 * Ngrid), **f)
    N_num = Ngrid[np.argmin(Lgrid)]
    assert abs(np.log(N_num / N_opt)) < 1e-3
    print(f"{name}: a = {a:.3f}, b = {1 - a:.3f}")
    print(f"  閉形式 N_opt = {N_opt / 1e9:6.1f}B, D_opt = {D_opt / 1e12:5.2f}T, D/N = {D_opt / N_opt:5.1f} トークン/パラメータ")
    print(f"  数値探索 N   = {N_num / 1e9:6.1f}B  ✔ 閉形式と一致")
    print(f"  70B / 1.4T で学習したときの予測損失 = {chin_loss(70e9, 1.4e12, **f):.4f}, "
          f"同じ計算量での最小値 = {chin_loss(N_opt, D_opt, **f):.4f}\n")

**IsoFLOP 曲線**：計算量 $C$ を固定して $N$ を動かすと（$D = C/6N$ も同時に動く）、損失は U 字型になります。$N$ が小さすぎるとモデルの容量不足、大きすぎるとデータ不足で損失が上がり、その間に最適点があります。最適点は $C$ とともに直線（両対数）上を動きます。

In [ ]:
f = fits["Besiroglu et al. (2024)"]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
Ngrid = np.logspace(7, 12.5, 400)
for C in [1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24]:
    Lc = chin_loss(Ngrid, C / (6 * Ngrid), **f)
    line, = axes[0].semilogx(Ngrid, Lc, label=f"C = {C:.0e}")
    N_opt, D_opt = chin_opt(C, **f)
    axes[0].plot(N_opt, chin_loss(N_opt, D_opt, **f), "o", color=line.get_color())
axes[0].set_ylim(1.8, 4.5); axes[0].set_xlabel("N (parameters)"); axes[0].set_ylabel("loss [nats]")
axes[0].set_title("IsoFLOP curves (Besiroglu et al. fit), dots = closed-form optimum"); axes[0].legend(fontsize=8)

Cs = np.logspace(18, 26, 50)
for name, fi in fits.items():
    N_o, D_o = chin_opt(Cs, **fi)
    axes[1].loglog(Cs, N_o, label=f"N_opt  {name}")
    axes[1].loglog(Cs, D_o, "--", label=f"D_opt  {name}")
axes[1].plot(C_chin, 70e9, "k*", ms=12, label="Chinchilla: 70B params")
axes[1].plot(C_chin, 1.4e12, "k^", ms=9, label="Chinchilla: 1.4T tokens")
axes[1].set_xlabel("C (FLOPs)"); axes[1].set_title("compute-optimal N and D")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

# GPT-2 small と同じ大きさのモデルにとっての計算最適データ量（N = G (N D)^a を D について解く）
a = f["beta"] / (f["alpha"] + f["beta"])
G = (f["alpha"] * f["A"] / (f["beta"] * f["B"])) ** (1 / (f["alpha"] + f["beta"]))
N_s = n_total
D_s = (N_s / G) ** (1 / a) / N_s
print(f"Besiroglu et al. の式で、N = {N_s / 1e6:.0f}M（GPT-2 small）の計算最適データ量 ≈ {D_s / 1e9:.1f}B トークン"
      f"（{D_s / N_s:.0f} トークン/パラメータ）")

**まとめ**
- Kaplan et al.：$N_{\text{opt}}\propto C^{0.73}$（モデルを大きく、データは控えめに）。
- Hoffmann et al.：$N_{\text{opt}}\propto C^{\approx 0.5}$（モデルとデータを同じ割合で）。実用上の目安として **約 20 トークン/パラメータ** が広く使われています。Hoffmann et al. 自身の式の定数をそのまま使うと上のように比がずれますが、Besiroglu et al. の再推定ではおよそ 20 になります。
- 両者の違いの原因として、学習率スケジュールを各設定に合わせて調整したかどうかや、パラメータ数の数え方（埋め込みを含むか）などが指摘されています。
- 「計算最適」は **学習** の計算量だけを見た基準です。推論コストまで考えると、小さいモデルを計算最適よりはるかに多いトークンで学習する方が有利なことも多く、近年の多くのモデルはそうしています。

## 12. まとめと参考文献

### このノートで確認したこと

| 節 | 確認したこと |
|---|---|
| 1 | パラメータ数 $= |\mathcal{V}|d + n_{\text{ctx}}d + L(12d^2+13d) + 2d = 124{,}439{,}808$。`attn.bias` は因果マスクのバッファ |
| 2 | バイトレベル BPE：未知語なし、可逆、結合過程、日本語は 1 文字あたりのトークン数が多い |
| 3 | 学習済み位置埋め込みに滑らかな波と帯状の類似度構造が現れる。トークン埋め込みの近傍 |
| 4–5 | Pre-LN ブロックを数式どおりに実装し、`transformers` と一致 |
| 6 | 1 回の順伝播で全位置の次トークン予測。自然な文は並べ替えた文より損失が小さい |
| 7 | attention sink、直前トークンヘッド、induction head（2 回目の繰り返しで損失が急落） |
| 8 | logit lens：正解トークン（Paris）は深い層で浮上する |
| 9 | 残差ストリームのノルムが層とともに増大し、中間層では先頭トークンだけ桁違いに大きい |
| 10 | KV キャッシュの一致と高速化、greedy の繰り返しと top-$k$ / top-$p$ |
| 11 | $N\approx 12Ld^2$、FLOPs の実測で $C_{\text{forward}}\approx 2N$、逆伝播は 2 倍で $C\approx 6ND$、Kaplan の冪乗則と Chinchilla の計算最適配分の導出 |

### 参考文献
- A. Radford, J. Wu, R. Child, D. Luan, D. Amodei, I. Sutskever. "Language Models are Unsupervised Multitask Learners." OpenAI Technical Report, 2019. [PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) / [openai/gpt-2](https://github.com/openai/gpt-2)
- J. Kaplan, S. McCandlish, T. Henighan, T. B. Brown, B. Chess, R. Child, S. Gray, A. Radford, J. Wu, D. Amodei. "Scaling Laws for Neural Language Models." 2020. [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
- J. Hoffmann, S. Borgeaud, A. Mensch, et al. "Training Compute-Optimal Large Language Models." NeurIPS 2022. [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
- T. Besiroglu, E. Erdil, M. Barnett, J. You. "Chinchilla Scaling: A replication attempt." 2024. [arXiv:2404.10102](https://arxiv.org/abs/2404.10102)
- 本文中で触れた現象：Holtzman et al. (2020) *The Curious Case of Neural Text Degeneration*（top-$p$ サンプリング）、Olsson et al. (2022) *In-context Learning and Induction Heads*、Xiao et al. (2023) *Efficient Streaming Language Models with Attention Sinks*、Sun et al. (2024) *Massive Activations in Large Language Models*、nostalgebraist (2020) *interpreting GPT: the logit lens*